# 📈 Notebook 04 — Results Dashboard
**CT Guanabara Futevolei | Data Science Performance Project**

Final results: pre vs post comparison, IPG evolution timelines,
and individualized recommendations for each athlete.


In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import matplotlib.patches as mpatches
from data_pipeline import load_raw, build_comparison, build_full_timeline, compute_ipg

raw  = load_raw()
comp = build_comparison()
tl   = build_full_timeline()

ATHLETES = comp['name'].tolist()
COLORS   = ['#1B4332','#E76F51','#2D6A4F','#74C69D','#B7E4C7']
METRICS  = ['jump_height_cm','ball_control_touches',
            'serve_accuracy_pct','reception_efficiency_pct','attack_rate_pct']

print("✅ Data loaded")
display(comp[['name','ipg_pre','ipg_post','ipg_delta_pct']].rename(columns={
    'name':'Athlete','ipg_pre':'IPG Pre','ipg_post':'IPG Post','ipg_delta_pct':'Δ IPG (%)'
}))


## 1. IPG Evolution — All Athletes (8-Week Timeline)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

for (aid, group), color in zip(tl.groupby('athlete_id'), COLORS):
    name  = group['name'].iloc[0]
    short = name.split()[0]
    ax.plot(group['week'], group['ipg_computed'], 'o-',
            color=color, lw=2.5, markersize=7, label=short)

ax.axvline(4.5, color='#6C757D', linestyle='--', lw=1.5, alpha=0.7,
           label='Intervention point')
ax.fill_betweenx([tl['ipg_computed'].min()-0.01, tl['ipg_computed'].max()+0.01],
                  0.5, 4.5, alpha=0.06, color='#1B4332', label='Pre-intervention')
ax.fill_betweenx([tl['ipg_computed'].min()-0.01, tl['ipg_computed'].max()+0.01],
                  4.5, 8.5, alpha=0.06, color='#E76F51', label='Post-intervention')

ax.set_xlabel('Week', fontsize=12)
ax.set_ylabel('IPG — General Performance Index', fontsize=12)
ax.set_title('IPG Evolution — All Athletes (8-Week Pilot)',
             fontsize=14, fontweight='bold', color='#1B4332')
ax.set_xticks(range(1, 9))
ax.set_xticklabels([f'W{i}' for i in range(1, 9)])
ax.legend(frameon=False, loc='upper left')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/figures/ipg_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: ipg_timeline.png")


## 2. Pre vs Post — All Metrics Comparison

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

metric_labels = ['Jump Height (cm)','Ball Control (touches)',
                 'Serve Accuracy (%)','Reception Efficiency (%)','Attack Rate (%)','IPG']
all_metrics   = METRICS + ['ipg']

names_short = [n.split()[0] for n in comp['name']]
x = np.arange(len(names_short)); w = 0.38

for i, (col, label) in enumerate(zip(all_metrics, metric_labels)):
    ax = axes[i]
    pre_vals  = comp[f'{col}_pre'].values
    post_vals = comp[f'{col}_post'].values
    b1 = ax.bar(x-w/2, pre_vals,  w, label='Pre',  color='#2D6A4F', alpha=0.85)
    b2 = ax.bar(x+w/2, post_vals, w, label='Post', color='#E76F51', alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(names_short, rotation=20, ha='right', fontsize=9)
    ax.set_title(label, fontweight='bold', fontsize=11)
    ax.legend(frameon=False, fontsize=9)
    ax.spines[['top','right']].set_visible(False)
    # delta labels
    for j, (pre_v, post_v) in enumerate(zip(pre_vals, post_vals)):
        delta = (post_v - pre_v) / pre_v * 100
        ax.text(j+w/2, post_v+0.5, f'+{delta:.1f}%', ha='center',
                fontsize=7.5, color='#1B4332', fontweight='bold')

plt.suptitle('Pre vs Post Intervention — All Performance Metrics',
             fontsize=15, fontweight='bold', color='#1B4332', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/pre_vs_post_all_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: pre_vs_post_all_metrics.png")


## 3. IPG Delta — Ranking by Improvement

In [ ]:
comp_sorted = comp.sort_values('ipg_delta_pct', ascending=True)
short_names = [n.split()[0] for n in comp_sorted['name']]

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#1B4332' if v == comp_sorted['ipg_delta_pct'].max() else '#52B788'
          for v in comp_sorted['ipg_delta_pct']]
bars = ax.barh(short_names, comp_sorted['ipg_delta_pct'], color=colors, edgecolor='white')
for bar, val in zip(bars, comp_sorted['ipg_delta_pct']):
    ax.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
            f'+{val:.1f}%', va='center', fontsize=10, color='#1B4332', fontweight='bold')
ax.set_xlabel('IPG Improvement (%)', fontsize=12)
ax.set_title('IPG Delta — Athletes Ranked by Improvement',
             fontsize=13, fontweight='bold', color='#1B4332')
ax.set_xlim(0, comp_sorted['ipg_delta_pct'].max() * 1.2)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/figures/ipg_delta_ranking.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Final Summary Table

In [ ]:
summary_out = comp[['name'] + [f'{m}_pre' for m in METRICS] +
                          [f'{m}_post' for m in METRICS] +
                          [f'{m}_delta_pct' for m in METRICS] +
                          ['ipg_pre','ipg_post','ipg_delta_pct']].copy()

summary_out.columns = (
    ['Athlete'] +
    [f'Pre_{m}' for m in ['JS','BC','SA','RE','AT']] +
    [f'Post_{m}' for m in ['JS','BC','SA','RE','AT']] +
    [f'Delta_{m}_pct' for m in ['JS','BC','SA','RE','AT']] +
    ['IPG_Pre','IPG_Post','IPG_Delta_pct']
)
summary_out.to_csv('../outputs/final_results_summary.csv', index=False)
print("✅ Saved: final_results_summary.csv")
display(summary_out[['Athlete','IPG_Pre','IPG_Post','IPG_Delta_pct']])


## 5. Radar Chart — Individual Athlete Profile (Pre vs Post)

In [ ]:
from matplotlib.patches import FancyArrowPatch

def radar_chart(ax, values_pre, values_post, labels, title, color):
    N = len(labels)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]
    v_pre  = list(values_pre) + [values_pre[0]]
    v_post = list(values_post) + [values_post[0]]
    ax.plot(angles, v_pre,  'o--', color=color, lw=1.5, alpha=0.6, label='Pre')
    ax.fill(angles, v_pre,  alpha=0.1, color=color)
    ax.plot(angles, v_post, 'o-',  color='#E76F51', lw=2, label='Post')
    ax.fill(angles, v_post, alpha=0.15, color='#E76F51')
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylim(0, 100); ax.set_title(title, fontweight='bold', pad=15, fontsize=10)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), frameon=False, fontsize=8)

labels = ['JS','BC','SA','RE','AT']
fig, axes = plt.subplots(2, 3, figsize=(18, 11), subplot_kw=dict(polar=True))
axes = axes.flatten()

for i, (_, row) in enumerate(comp.iterrows()):
    pre_v  = [row[f'{m}_pre']  for m in METRICS]
    post_v = [row[f'{m}_post'] for m in METRICS]
    # normalize BC to 0-100 scale
    pre_v[1]  = pre_v[1]  / 80 * 100
    post_v[1] = post_v[1] / 80 * 100
    radar_chart(axes[i], pre_v, post_v, labels,
                row['name'].split()[0], COLORS[i % len(COLORS)])

axes[-1].axis('off')  # hide last empty subplot
plt.suptitle('Athlete Performance Radar — Pre vs Post Intervention',
             fontsize=14, fontweight='bold', color='#1B4332', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/radar_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: radar_charts.png")


## ✅ Project Complete!

All analysis notebooks have been executed. Check `outputs/figures/` for all generated charts and `outputs/` for CSV summaries.